In [30]:
!
import kagglehub

# Download latest version
path = kagglehub.dataset_download("amirmahdiabbootalebi/heart-disease")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'heart-disease' dataset.
Path to dataset files: /kaggle/input/heart-disease


In [31]:
import torch
import pandas as pd

In [32]:
heart = pd.read_csv(f'{path}/heart.csv')
print(heart.shape)
print(heart.columns)
print(heart)

(918, 12)
Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope',
       'HeartDisease'],
      dtype='object')
     Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  \
0     40   M           ATA        140          289          0     Normal   
1     49   F           NAP        160          180          0     Normal   
2     37   M           ATA        130          283          0         ST   
3     48   F           ASY        138          214          0     Normal   
4     54   M           NAP        150          195          0     Normal   
..   ...  ..           ...        ...          ...        ...        ...   
913   45   M            TA        110          264          0     Normal   
914   68   M           ASY        144          193          1     Normal   
915   57   M           ASY        130          131          0     Normal   
916   57   F           ATA        

In [33]:
heart_encoded = pd.get_dummies(heart, columns=['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope'], drop_first=True)
heart_data = torch.tensor(heart_encoded.astype(float).to_numpy(), dtype=torch.float32)
print(heart_data)
print(heart_data.shape)

tensor([[ 40., 140., 289.,  ...,   0.,   0.,   1.],
        [ 49., 160., 180.,  ...,   0.,   1.,   0.],
        [ 37., 130., 283.,  ...,   0.,   0.,   1.],
        ...,
        [ 57., 130., 131.,  ...,   1.,   1.,   0.],
        [ 57., 130., 236.,  ...,   0.,   1.,   0.],
        [ 38., 138., 175.,  ...,   0.,   0.,   1.]])
torch.Size([918, 16])


Check the cleanness of the data

In [34]:
print('NAN?', torch.isnan(heart_data).any())
print('inf?', torch.isinf(heart_data).any())

NAN? tensor(False)
inf? tensor(False)


Let's do some distribution analysis

In [35]:
print(f'Mean: {torch.mean(heart_data)}')
print(f'Min: {torch.min(heart_data)}')
print(f'Max: {torch.max(heart_data)}')
print(f'Std: {torch.max(heart_data)}')

Mean: 32.910648345947266
Min: -2.5999999046325684
Max: 603.0
Std: 603.0


Now let's split the data into features and target variable

In [36]:
X = heart_data[:, :-1]
Y = heart_data[:, -1]
print(X.shape)
print(Y.shape)

torch.Size([918, 15])
torch.Size([918])


Now let's split the dataset into training and test

In [37]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=0.2, random_state=42)

In [38]:
print(f"X_train: {X_train.shape} X_test: {X_test.shape}")
print(f"Y_train; {Y_train.shape} Y_test: {Y_test.shape}")

X_train: torch.Size([734, 15]) X_test: torch.Size([184, 15])
Y_train; torch.Size([734]) Y_test: torch.Size([184])


In [39]:
import torch.nn as nn

class LinearRegressionModel(nn.Module):
  def __init__(self,in_features, out_features):
    super().__init__()
    self.linear_layer = nn.Linear(in_features, out_features)
    self.sigmoid = nn.Sigmoid()

  def forward(self,x):
    x = self.linear_layer(x)
    x = self.sigmoid(x)
    return x

# Initiating the model
model = LinearRegressionModel(in_features=15,out_features=1)
print(model)



LinearRegressionModel(
  (linear_layer): Linear(in_features=15, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


Now let's select our **optimizer** and ***loss function***

In [40]:
import torch.optim as optim

learning_rate = 0.01

optimizer = optim.Adam(model.parameters(), lr = learning_rate)
# Binary Cross Entropy for  classification
loss_fun = nn.BCELoss()

In [41]:
epochs = 1000

for epoch in range(epochs):
  y_hat = model(X_train)
  # Loss calcualtion
  loss = loss_fun(y_hat, Y_train.unsqueeze(1))

  # The three line mantra
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()

  if epoch % 100 == 0:
    print(f'Epoch: {epoch}  Loss: {loss.item():.4f}')

Epoch: 0  Loss: 58.1744
Epoch: 100  Loss: 58.1744
Epoch: 200  Loss: 58.1744
Epoch: 300  Loss: 58.1744
Epoch: 400  Loss: 58.1744
Epoch: 500  Loss: 58.1744
Epoch: 600  Loss: 58.1744
Epoch: 700  Loss: 58.1744
Epoch: 800  Loss: 58.1744
Epoch: 900  Loss: 58.1744


#Evaluation

In [42]:
model.eval()
with torch.inference_mode():
  y_pred = model(X_test)
  test_loss = loss_fun(y_pred, Y_test.unsqueeze(1))

print(f'Test Loss: {test_loss.item():.4f}')

Test Loss: 52.1739


In [43]:
from sklearn.metrics import accuracy_score

y_pred_class = (y_pred >=0.5).float()

accuracy = accuracy_score(Y_test.cpu().numpy(), y_pred_class.cpu().numpy())
print(f"the model's accuacy is: {accuracy * 100:.2f}%" )

the model's accuacy is: 47.83%


Let's perform inference using the model

In [44]:
eg_data = torch.tensor([45.0, 120.0, 200.0, 0.0, 150.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0,0.0,0.0,1.0,0.0], dtype= torch.float32).unsqueeze(0)
model.eval()
with torch.inference_mode():
  new_prediction_prob = model(eg_data)
  new_pre_class = (new_prediction_prob >= 0.5).float()

  print(f'New data prediction: {new_pre_class}')

New data prediction: tensor([[1.]])
